# Tạo dataset `.npy` cho ASVspoof 2019 LA (chạy MỘT lần, CPU)

Input: dataset ASVspoof 2019 LA gốc. Accelerator: None (CPU). Internet: bật.
Chạy **Save Version → Save & Run All**, rồi **New Dataset từ output** của version này.
Output (khoảng 13 GB): `protocols/`, `train/`, `dev/`, `eval/` (`<file_id>.npy` int16), `asv_scores/`.
`load_audio()` đọc `.npy` cho kết quả giống hệt FLAC.

In [ ]:
# ===== Cell 0 — THAM SỐ =====
REPO_URL = "https://github.com/Phvster/antispoof-benchmark.git"
REPO_REF = "__REPO_REF__"          # hash commit cố định
LA = "/kaggle/input/asvspoof2019-la/LA"   # TÊN DATASET LÀ THAM SỐ: thư mục chứa ASVspoof2019_LA_*
OUT = "/kaggle/working"
REPO = "/tmp/antispoof-benchmark"  # ngoài output

In [ ]:
# ===== Cell 1 — clone repo (token chỉ-đọc từ Kaggle Secrets: GITHUB_TOKEN) =====
import os, subprocess, sys, shutil

def sh(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

def clone_repo():
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print("Không đọc được secret GITHUB_TOKEN (repo public thì bỏ qua):", type(e).__name__)
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@", 1) if token else REPO_URL
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError("git clone lỗi (kiểm tra secret GITHUB_TOKEN):\n" + msg)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", REPO_URL], check=True)

if not os.path.exists(REPO):
    clone_repo()
sh(["git", "-C", REPO, "checkout", "-q", REPO_REF])
sh(["git", "-C", REPO, "log", "-1", "--oneline"])
sh([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements.txt"])
os.chdir(REPO)

In [ ]:
# ===== Cell 2 — protocol chung (in md5 -> ghi vào protocols/README.md) =====
sh([sys.executable, "-m", "protocols.convert_asvspoof2019_la",
    "--src", f"{LA}/ASVspoof2019_LA_cm_protocols", "--out_dir", f"{OUT}/protocols"])

In [ ]:
# ===== Cell 3 — giải mã FLAC -> .npy int16 (bỏ qua file đã có) =====
for s in ("train", "dev", "eval"):
    sh([sys.executable, "-m", "tools.decode_to_npy",
        "--protocol", f"{OUT}/protocols/asvspoof2019_la_{s}.csv",
        "--audio_dir", f"{LA}/ASVspoof2019_LA_{s}/flac", "--out_dir", f"{OUT}/{s}",
        "--workers", str(os.cpu_count())])

In [ ]:
# ===== Cell 4 — file điểm ASV của BTC (cho min t-DCF) + kiểm tra =====
asv = f"{LA}/ASVspoof2019_LA_asv_scores/ASVspoof2019.LA.asv.eval.gi.trl.scores.txt"
os.makedirs(f"{OUT}/asv_scores", exist_ok=True)
if os.path.exists(asv):
    shutil.copy(asv, f"{OUT}/asv_scores/")
    print("đã chép", asv)
else:
    print("KHÔNG thấy", asv, "-> t-DCF sẽ cần gắn thêm dataset LA gốc")

import numpy as np
from common.audio import load_audio
from common.data import read_protocol
for s in ("train", "dev", "eval"):
    df = read_protocol(f"{OUT}/protocols/asvspoof2019_la_{s}.csv")
    n = sum(os.path.exists(f"{OUT}/{s}/{f}.npy") for f in df["file_id"])
    print(s, f"{n}/{len(df)} file .npy")
    assert n == len(df)
    f = df["file_id"].iloc[0]
    a, b = load_audio(f"{LA}/ASVspoof2019_LA_{s}/flac/{f}.flac"), load_audio(f"{OUT}/{s}/{f}.npy")
    assert np.array_equal(a, b), f"{s}/{f}: FLAC khác NPY"
sh(["du", "-sh", OUT])